<a href="https://colab.research.google.com/github/ofer-geo/GTFS-Data-Agent/blob/main/AI_Agents_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1: Define 'brain' (provider + model)

Cell 2: Install & import (pip install groq, streamlit, etc.)

Cell 3: Create the Groq client

Cell 4: Tools (the two functions: get_open_bus_endpoints, query_open_bus_api)

Cell 5: tools_map

Cell 6: tools_schema

Cell 7: system_prompt

Cell 8: get_messages

Cell 9: react_agent (the simple version for testing in the notebook)

Cell 10: Test the agent (run a question directly)

Cell 11: Build agent_core.py (writes the file for Streamlit)

Cell 12: Append SYSTEM_PROMPT + tools_schema to agent_core.py

Cell 13: Write app.py

Cell 14: Run Streamlit + cloudflared

In [1]:
## Setup

In [2]:
!pip install -q openai anthropic groq langchain langchain-openai langchain-anthropic langchain-groq
!pip install -q requests wikipedia-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.8/923.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 3.9 MB/s eta 0:00:00


# 1. Define 'brain'



Grok is for free - register in site https://groq.com/




In [3]:
import os
import json
from google.colab import userdata

# Choose Provider
PROVIDER = "groq"  # Changable

# load API Key
if PROVIDER == "groq":
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    MODEL = "llama-3.3-70b-versatile"
elif PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    MODEL = "gpt-4o-mini"
elif PROVIDER == "anthropic":
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    MODEL = "claude-sonnet-4-20250514"

print(f"Using {PROVIDER} with model {MODEL}")


Using groq with model llama-3.3-70b-versatile


### Grok models

In [4]:
from groq import Groq
client = Groq()

models = client.models.list()
print("Available Groq models:")
print("-" * 40)
for m in models.data:
    print(m.id)

Available Groq models:
----------------------------------------
openai/gpt-oss-20b
whisper-large-v3-turbo
openai/gpt-oss-safeguard-20b
meta-llama/llama-4-scout-17b-16e-instruct
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
whisper-large-v3
llama-3.1-8b-instant
canopylabs/orpheus-arabic-saudi
llama-3.3-70b-versatile
qwen/qwen3.6-27b
qwen/qwen3-32b
groq/compound-mini
meta-llama/llama-prompt-guard-2-86m
groq/compound
canopylabs/orpheus-v1-english
allam-2-7b


In [5]:
print(f"Current MODEL: {MODEL}")

Current MODEL: llama-3.3-70b-versatile


### Check connection

 Lets test the connection and basic functionality of different LLM providers

In [6]:
from groq import Groq
from openai import OpenAI
from anthropic import Anthropic

try:
  if PROVIDER == "groq":
      client = Groq()
      response = client.chat.completions.create(
          model="llama-3.1-8b-instant",
          messages=[{"role": "user", "content": "Say 'Hello' in Hebrew"}],
          max_tokens=50
      )
      result = response.choices[0].message.content

  elif PROVIDER == "openai":
      client = OpenAI()
      response = client.chat.completions.create(
          model="gpt-4o-mini",
          messages=[{"role": "user", "content": "Say 'Hello' in Hebrew"}],
          max_tokens=50
      )
      result = response.choices[0].message.content

  elif PROVIDER == "anthropic":
      client = Anthropic()
      response = client.messages.create(
          model="claude-3-5-haiku-20241022",
          max_tokens=50,
          messages=[{"role": "user", "content": "Say 'Hello' in Hebrew"}]
      )
      result = response.content[0].text

  print(f"result: {result}")

except Exception as e:
  print(f" Error occoured:  {e}")

result: "Shalom" is a common way to say "hello" or "goodbye" in Hebrew. It roughly translates to "peace" and is often used as a greeting. A more direct way to say "hello" in Hebrew is "" (


# 2. Install & Import

In [7]:
!pip install -q groq streamlit

import os
import json
import re
import requests
from groq import Groq, BadRequestError, APIStatusError

print("Imports OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 32.8 MB/s eta 0:00:00
Imports OK


# 3. Create the Groq Client


In [8]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Client ready")

Client ready


# 4. Tools

In [9]:
OPEN_BUS_BASE_URL = "https://open-bus-stride-api.hasadna.org.il"
HEADERS = {
    "User-Agent": "AgenticAI-Course/1.0 (Educational; Bar-Ilan University)",
    "Accept": "application/json",
}


def get_open_bus_endpoints(filter_keyword: str = "") -> str:
    """Discover available Open Bus API endpoints and their query parameters."""
    url = OPEN_BUS_BASE_URL + "/openapi.json"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        paths = resp.json().get("paths", {})
        lines = []
        for path, methods in sorted(paths.items()):
            if filter_keyword and filter_keyword.lower() not in path.lower():
                continue
            get = methods.get("get")
            if not get:
                continue
            params = [p.get("name") for p in get.get("parameters", [])]
            lines.append(f"{path}\n    params: {', '.join(params) if params else '(none)'}")
        return "\n".join(lines) if lines else f"No endpoints matched '{filter_keyword}'."
    except Exception as e:
        return f"Error fetching API spec: {e}"


def query_open_bus_api(endpoint: str, params_json: str = "{}") -> str:
    """Query an Open Bus STRIDE API list endpoint. params_json is a JSON object string."""
    try:
        params = json.loads(params_json) if params_json else {}
    except json.JSONDecodeError:
        return f"Invalid params_json (must be a JSON object string): {params_json!r}"
    try:
        resp = requests.get(OPEN_BUS_BASE_URL + endpoint, params=params, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        if isinstance(data, list):
            preview = data[:5]
            return (f"Returned {len(data)} record(s) (showing first {len(preview)}).\n"
                    + json.dumps(preview, ensure_ascii=False, default=str))
        return json.dumps(data, ensure_ascii=False, default=str)
    except requests.exceptions.HTTPError as e:
        body = resp.text[:500] if "resp" in dir() else ""
        return f"HTTP error: {e}. Response body: {body}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {e}"
    except Exception as e:
        return f"Unexpected error: {e}"


print("Tools defined")

Tools defined


# 5. Tools Map

In [10]:
tools_map = {
    "get_open_bus_endpoints": get_open_bus_endpoints,
    "query_open_bus_api": query_open_bus_api,
}

print("tools_map ready:", list(tools_map.keys()))

tools_map ready: ['get_open_bus_endpoints', 'query_open_bus_api']


# 6. Tools Schema

In [11]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_open_bus_endpoints",
            "description": (
                "Discover the available Open Bus STRIDE API endpoints and their exact query "
                "parameter names by reading the live OpenAPI spec. Use this FIRST whenever you "
                "are unsure which endpoint or parameter to use, instead of guessing."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "filter_keyword": {
                        "type": "string",
                        "description": (
                            "Optional substring to filter endpoints by path, "
                            "e.g. 'siri', 'gtfs', 'vehicle', 'ride'. Leave empty to list all."
                        ),
                    }
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "query_open_bus_api",
            "description": (
                "Query an Open Bus STRIDE API list endpoint (Israeli public transport: "
                "real-time SIRI data and planned GTFS data). Returns JSON records. "
                "Use for bus stops, ride counts, live vehicle locations, and speed."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "endpoint": {
                        "type": "string",
                        "description": (
                            "The API path to call, e.g. '/siri_vehicle_locations/list', "
                            "'/gtfs_ride_stops/list', '/gtfs_rides/list'."
                        ),
                    },
                    "params_json": {
                        "type": "string",
                        "description": (
                            "Query parameters as a JSON OBJECT STRING (not a nested object). "
                            "Example: '{\"gtfs_route__route_short_name\": \"189\", \"limit\": 5}'. "
                            "Use '{}' for no parameters."
                        ),
                    },
                },
                "required": ["endpoint"],
            },
        },
    },
]

print("tools_schema ready:", len(tools_schema), "tools")

tools_schema ready: 2 tools


# 7. System Prompt

In [35]:
system_prompt = """You are a public-transport data assistant for Israel, using the Open Bus STRIDE API.

KEY ENDPOINTS:
- /gtfs_ride_stops/list : stops of a line. Filter: gtfs_route__route_short_name (the line number), arrival_time_from + arrival_time_to (a full day window, REQUIRED, e.g. "2023-01-01T00:00:00Z".."2023-01-01T23:59:59Z"), order_by="stop_sequence asc". First stop = row with stop_sequence=1. Each row already has gtfs_stop__name, gtfs_stop__lat, gtfs_stop__lon.
- /gtfs_rides/list : count rides of a line. Use get_count=true. Filter: gtfs_route__route_short_name, start_time_from, start_time_to.
- /siri_vehicle_locations/list : live position + speed (lat, lon, velocity, recorded_at_time). Bound with recorded_at_time_from/to and small limit.

TOOLS:
- get_open_bus_endpoints(filter_keyword): list real endpoints + exact param names. Use when unsure.
- query_open_bus_api(endpoint, params_json): params_json is a JSON object STRING, e.g. '{"gtfs_route__route_short_name":"189","limit":5}'.

RULES:
1. One tool call at a time. Wait for the result before the next.
2. NEVER claim a question is "too large" or give up unless a tool actually returned an error. On timeout, just retry the same query.
3. Keep queries small: narrow time windows, small limit, get_count for counting.
4. For maps, list coordinates as {lat, lon, label} in your final answer.
5. CANNOT answer: actual delay (data often empty), passenger counts (not in Open Bus). Say so if asked.
6. Answer in the user's language (Hebrew if asked in Hebrew). State assumptions (date, direction).
7. For "list stops" or "show on map" questions: fetch ONCE from /gtfs_ride_stops/list with order_by="stop_sequence asc" and limit=50. Do NOT call it repeatedly with bigger limits.
8. When showing stops on a map, end your answer with a clean JSON array of OBJECTS (not quoted strings): [{"lat":31.80,"lon":35.10,"label":"name"},{"lat":31.79,"lon":35.10,"label":"name"}]. Do NOT wrap each object in quotes or escape the inner quotes. Use gtfs_stop__lat and gtfs_stop__lon for the values.
"""

print("system_prompt ready,", len(system_prompt), "chars")

system_prompt ready, 2037 chars


# 8. Get Messages

In [13]:
def get_messages(question: str):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    return messages

print("get_messages ready")

get_messages ready


# 9. ReAct Agent (notebook test version)

In [14]:
def react_agent(question: str, max_steps: int = 15):
    print(f"\nquestion: {question}")
    print("-" * 50)

    messages = get_messages(question)
    MAX_OBS_CHARS = 3000
    tool_calls_made = 0

    for step in range(max_steps):
        print(f"\nstep {step + 1}:")

        try:
            response = client.chat.completions.create(
                model=MODEL, messages=messages, tools=tools_schema,
                tool_choice="auto", parallel_tool_calls=False)
        except BadRequestError as e:
            if "tool_use_failed" in str(e):
                print(" (tool_use_failed - retrying)")
                messages.append({"role": "user", "content":
                    "Your previous tool call was malformed. Call exactly ONE tool using the "
                    "structured function-calling format, NOT as plain text. Try again."})
                continue
            raise
        except APIStatusError as e:
            es = str(e)
            if getattr(e, "status_code", None) in (413, 429) or "rate_limit" in es or "too large" in es.lower():
                import time
                print(" (rate limit hit - waiting 20s and retrying...)")
                time.sleep(20)
                continue
            raise

        msg = response.choices[0].message

        if not msg.tool_calls:
            # Guard: if the model tries to give up before ever using a tool, force it to try.
            if tool_calls_made == 0:
                print(" (model tried to answer without using any tool - forcing it to try)")
                messages.append({"role": "user", "content":
                    "Do NOT give up and do NOT claim any size limit. You have NOT called any tool yet. "
                    "Use get_open_bus_endpoints or query_open_bus_api now to actually look up the answer."})
                continue
            print(f"\nThought: I have enough information to answer")
            print(f"\nfinal answer:\n{msg.content}")
            return msg.content

        messages.append(msg)

        for tool_call in msg.tool_calls:
            func_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            print(f" Action: {func_name}({json.dumps(args, ensure_ascii=False)})")

            result = tools_map[func_name](**args)
            tool_calls_made += 1
            print(f" Observation: {result[:200]}..." if len(result) > 200 else f" Observation: {result}")

            trimmed = result if len(result) <= MAX_OBS_CHARS else result[:MAX_OBS_CHARS] + "\n...[truncated]"
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": trimmed
            })

    return "Max steps reached"

print("react_agent ready (with no-giveup guard)")

react_agent ready (with no-giveup guard)


# 10. Test the Agent

In [15]:
react_agent("What is the first stop of line 189?")


question: What is the first stop of line 189?
--------------------------------------------------

step 1:
 Action: query_open_bus_api({"endpoint": "/gtfs_ride_stops/list", "params_json": "{\"gtfs_route__route_short_name\": \"189\", \"arrival_time_from\": \"2024-04-08T00:00:00Z\", \"arrival_time_to\": \"2024-04-08T23:59:59Z\", \"order_by\": \"stop_sequence asc\", \"limit\": 1}"})
 Observation: Returned 1 record(s) (showing first 1).
[{"id": 2585941699, "gtfs_stop_id": 22835238, "gtfs_ride_id": 72340839, "arrival_time": "2024-04-08T04:55:00+00:00", "departure_time": "2024-04-08T04:55:00+00:0...

step 2:

Thought: I have enough information to answer

final answer:
The first stop of line 189 is "מרכז לגיל הרך/הרב א.גורדון" (Mercaz Lagil Hara-ch/ Harav A. Gordon) located at {31.804551, 35.104361}.


'The first stop of line 189 is "מרכז לגיל הרך/הרב א.גורדון" (Mercaz Lagil Hara-ch/ Harav A. Gordon) located at {31.804551, 35.104361}.'

In [16]:
try:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "say hi"}],
        max_tokens=10,
    )
    print("OK:", r.choices[0].message.content)
except Exception as e:
    print("ERROR TYPE:", type(e).__name__)
    print("ERROR:", str(e)[:500])

OK: Hi, how are you today?


# 11. Build agent_core.py

In [39]:
%%writefile agent_core.py
import os, json, re, time, requests
from groq import Groq, BadRequestError, APIStatusError

MODEL = "llama-3.3-70b-versatile"
client = Groq(api_key=os.environ.get("GROQ_API_KEY", ""))

OPEN_BUS_BASE_URL = "https://open-bus-stride-api.hasadna.org.il"
HEADERS = {
    "User-Agent": "AgenticAI-Course/1.0 (Educational; Bar-Ilan University)",
    "Accept": "application/json",
}


def query_open_bus_api(endpoint: str, params_json: str = "{}") -> str:
    try:
        params = json.loads(params_json) if params_json else {}
    except json.JSONDecodeError:
        return f"Invalid params_json: {params_json!r}"
    try:
        resp = requests.get(OPEN_BUS_BASE_URL + endpoint, params=params, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        if isinstance(data, list):
            preview = data[:5]
            return (f"Returned {len(data)} record(s) (showing first {len(preview)}).\n"
                    + json.dumps(preview, ensure_ascii=False, default=str))
        return json.dumps(data, ensure_ascii=False, default=str)
    except requests.exceptions.HTTPError as e:
        body = resp.text[:500] if "resp" in dir() else ""
        return f"HTTP error: {e}. Body: {body}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {e}"
    except Exception as e:
        return f"Unexpected error: {e}"


tools_map = {"query_open_bus_api": query_open_bus_api}


def extract_coords(text):
    coords = []
    # Find each {...} block and try to parse lat/lon/label from it individually
    for block in re.findall(r'\{[^{}]+\}', text):
        # unescape common mangling: \" -> "
        clean = block.replace('\\"', '"').replace("\\", "")
        lat_m = re.search(r'"?lat"?\s*:\s*(-?\d{1,2}\.\d+)', clean)
        lon_m = re.search(r'"?lon"?\s*:\s*(-?\d{1,3}\.\d+)', clean)
        if not (lat_m and lon_m):
            continue
        lat, lon = float(lat_m.group(1)), float(lon_m.group(1))
        if not (29 < lat < 34 and 34 < lon < 36):
            continue
        label_m = re.search(r'"?label"?\s*:\s*"?([^"}]+)', clean)
        label = label_m.group(1).strip().strip('"') if label_m else ""
        coords.append({"lat": lat, "lon": lon, "label": label})
    # de-dup by rounded coordinate
    seen, out = set(), []
    for c in coords:
        key = (round(c["lat"], 5), round(c["lon"], 5))
        if key not in seen:
            seen.add(key)
            out.append(c)
    return out


def get_messages(question):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]


def react_agent_ui(question, max_steps=15):
    messages = get_messages(question)
    log, coords = [], []
    MAX_OBS_CHARS = 1200
    for step in range(max_steps):
        try:
            response = client.chat.completions.create(
                model=MODEL, messages=messages, tools=tools_schema,
                tool_choice="auto", parallel_tool_calls=False)
        except BadRequestError as e:
            if "tool_use_failed" in str(e):
                log.append({"type": "retry", "text": "tool_use_failed - retrying"})
                yield {"status": "retry", "log": list(log), "coords": list(coords), "answer": None}
                messages.append({"role": "user", "content":
                    "Your previous tool call was malformed. Call exactly ONE tool using the "
                    "structured function-calling format, NOT as plain text. Try again."})
                continue
            raise
        except APIStatusError as e:
            es = str(e)
            if getattr(e, "status_code", None) in (413, 429) or "rate_limit" in es or "too large" in es.lower():
                log.append({"type": "retry", "text": "rate limit - waiting 20s"})
                yield {"status": "retry", "log": list(log), "coords": list(coords), "answer": None}
                time.sleep(20)
                continue
            raise
        msg = response.choices[0].message
        if not msg.tool_calls:
            content = msg.content or ""
            # Detect a tool call that leaked into text instead of being a real call
            if '"name": "query_open_bus_api"' in content or '"type": "function"' in content:
                log.append({"type": "retry", "text": "model emitted a tool call as text - retrying"})
                yield {"status": "retry", "log": list(log), "coords": list(coords), "answer": None}
                messages.append({"role": "user", "content":
                    "You wrote a tool call as plain text. That does NOT work. Make a REAL function "
                    "call using the API's tool-calling mechanism, OR if you already have the data, "
                    "give the final answer in plain language with a coordinate list."})
                continue
            coords += extract_coords(content)
            yield {"status": "done", "log": list(log), "coords": list(coords), "answer": content}
            return
        messages.append(msg)
        for tool_call in msg.tool_calls:
            func_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            yield {"status": "calling", "tool": func_name, "args": args,
                   "log": list(log), "coords": list(coords), "answer": None}
            if func_name not in tools_map:
                result = f"Error: tool '{func_name}' does not exist. The only available tool is query_open_bus_api."
            else:
                result = tools_map[func_name](**args)
            log.append({"type": "action", "tool": func_name, "args": args, "observation": result[:500]})
            coords += extract_coords(result)
            yield {"status": "step", "log": list(log), "coords": list(coords), "answer": None}
            trimmed = result if len(result) <= MAX_OBS_CHARS else result[:MAX_OBS_CHARS] + "\n...[truncated]"
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": trimmed})
    yield {"status": "done", "log": list(log), "coords": list(coords), "answer": "Max steps reached"}

Overwriting agent_core.py


# 12. Append SYSTEM_PROMPT + tools_schema to agent_core.py

In [40]:
with open("agent_core.py", "a") as f:
    f.write("\n\nSYSTEM_PROMPT = " + repr(system_prompt) + "\n")
    f.write("tools_schema = " + repr(tools_schema) + "\n")

import importlib, agent_core
importlib.reload(agent_core)
print("react_agent_ui exists?", hasattr(agent_core, "react_agent_ui"))
print("SYSTEM_PROMPT exists?", hasattr(agent_core, "SYSTEM_PROMPT"))
print("tools_schema exists?", hasattr(agent_core, "tools_schema"))

react_agent_ui exists? True
SYSTEM_PROMPT exists? True
tools_schema exists? True


# 13. Write app.py

In [44]:
%%writefile app.py
import streamlit as st
import pandas as pd
from agent_core import react_agent_ui

st.set_page_config(page_title="Open Bus Agent", page_icon="🚌", layout="wide")

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

:root {
    --accent: #4285f4;
    --accent-soft: #e8f0fe;
    --ink: #202124;
    --muted: #5f6368;
    --line: #e8eaed;
    --bg: #ffffff;
    --sidebar: #f8f9fa;
}

html, body, [class*="css"], .stApp { font-family: 'Inter', sans-serif; }
.stApp { background: var(--bg); }
h1,h2,h3,h4,h5,h6 { color: var(--ink); font-weight: 700; letter-spacing: -0.01em; }
p, span, label, div { color: var(--ink); }
.stCode, code, pre { text-align: left; }

/* Sidebar */
[data-testid="stSidebar"] { background: var(--sidebar); border-right: 1px solid var(--line); }
[data-testid="stSidebar"] h3 { font-size: 13px; text-transform: uppercase; letter-spacing: 0.05em; color: var(--muted); font-weight: 600; }

/* Page title */
.page-title { font-size: 32px; font-weight: 800; color: var(--ink); margin: 4px 0 2px 0; }
.page-sub { font-size: 15px; color: var(--muted); margin-bottom: 28px; }

/* Section label */
.section { font-size: 20px; font-weight: 700; color: var(--ink); margin: 26px 0 4px 0; }

/* Metric row */
.metric-label { font-size: 13px; color: var(--muted); font-weight: 500; margin-bottom: 2px; }
.metric-value { font-size: 30px; font-weight: 700; color: var(--ink); }

/* Divider */
hr { border: none; border-top: 1px solid var(--line); margin: 22px 0; }

/* Buttons */
.stButton button {
    text-align: left; border-radius: 8px; font-weight: 500; font-size: 14px;
    border: 1px solid var(--line); background: #fff; color: var(--ink);
    transition: all .12s ease;
}
.stButton button:hover { border-color: var(--accent); color: var(--accent); background: var(--accent-soft); }
.stButton button[kind="primary"] { background: var(--accent); color: #fff; border: none; font-weight: 600; }

/* Input */
.stTextInput input {
    border-radius: 8px; border: 1.5px solid var(--line); padding: 10px 14px; font-size: 15px;
}
.stTextInput input:focus { border-color: var(--accent); box-shadow: 0 0 0 3px var(--accent-soft); }

/* Step timeline */
.step { border-left: 3px solid var(--accent); background: var(--accent-soft);
        border-radius: 6px; padding: 7px 12px; margin: 5px 0; font-size: 13px; color: var(--ink); }
.step-retry { border-left-color: #f9ab00; background: #fef7e0; }
.step-done { border-left-color: #34a853; background: #e6f4ea; }
</style>
""", unsafe_allow_html=True)

# --- Header ---
st.markdown("""
<div style="display:flex;align-items:center;gap:14px;">
    <div style="font-size:42px;">🚍</div>
    <div>
        <div class="page-title">Open Bus Agent</div>
        <div class="page-sub">Ask in plain language about stops, ride counts, and bus locations — powered by the Open Bus STRIDE API</div>
    </div>
</div>
""", unsafe_allow_html=True)

EXAMPLES = [
    "What is the first stop of line 189?",
    "What is the first stop of line 480?",
    "How many rides does line 19 have between 07:00 and 08:00 on Jan 1, 2023?",
    "How many rides does line 18 have on Jan 1, 2023?",
    "Show the first 5 stops of line 189 on a map",
]

CANNOT = [
    "Real-time delay — data often unmatched / unreliable",
    "Passenger counts / occupancy — not in Open Bus",
    "Long time ranges — exceed the token limit",
]

with st.sidebar:
    st.markdown("### Example questions")
    for ex in EXAMPLES:
        if st.button(ex, use_container_width=True):
            st.session_state["q"] = ex
    st.markdown("---")
    st.markdown("### Out of scope")
    for c in CANNOT:
        st.markdown(f"<div style='color:#5f6368;font-size:13px;margin:5px 0;'>• {c}</div>",
                    unsafe_allow_html=True)

st.markdown("<hr/>", unsafe_allow_html=True)
st.markdown("<div class='section'>Ask the agent</div>", unsafe_allow_html=True)

q = st.text_input("", value=st.session_state.get("q", ""),
                  placeholder="e.g. What is the first stop of line 189?",
                  label_visibility="collapsed")
go = st.button("Run query", type="primary")

if go and q.strip():
    status_box = st.empty()
    log_box = st.empty()
    final_answer, final_coords, final_log = None, [], []

    for update in react_agent_ui(q):
        final_log = update.get("log", [])
        final_coords = update.get("coords", [])

        if update["status"] == "calling":
            status_box.markdown(
                f"<div class='step'>🔧 Step {len(final_log)+1}: calling {update['tool']} ...</div>",
                unsafe_allow_html=True)
        elif update["status"] == "step":
            status_box.markdown(
                f"<div class='step'>✓ {len(final_log)} steps done, continuing...</div>",
                unsafe_allow_html=True)
        elif update["status"] == "retry":
            status_box.markdown(
                "<div class='step step-retry'>⟳ Retrying...</div>", unsafe_allow_html=True)

        with log_box.container():
            with st.expander("Agent steps (ReAct log)", expanded=True):
                for i, s in enumerate(final_log, 1):
                    if s["type"] == "retry":
                        st.markdown(f"<div class='step step-retry'>{i}. ⟳ {s['text']}</div>",
                                    unsafe_allow_html=True)
                    else:
                        st.markdown(f"<div class='step'><b>{i}. {s['tool']}</b></div>",
                                    unsafe_allow_html=True)
                        st.code(str(s["args"]), language="json")
                        st.text(s["observation"])

    status_box.markdown("<div class='step step-done'>✅ Done</div>", unsafe_allow_html=True)
    final_answer = update.get("answer")

    st.markdown("<div class='section'>💬 Answer</div>", unsafe_allow_html=True)
    st.write(final_answer)

    if final_coords:
        st.markdown("<div class='section'>🗺️ Map</div>", unsafe_allow_html=True)
        df = pd.DataFrame(final_coords)
        st.map(df[["lat", "lon"]])
        with st.expander("📍 Map points"):
            st.dataframe(df, use_container_width=True)

Overwriting app.py


# 14. Run Streamlit + cloudflared

In [45]:
import os
from google.colab import userdata

# Make sure the key is in the environment for the Streamlit process
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Stop any old processes
!pkill -f streamlit 2>/dev/null
!pkill -f cloudflared 2>/dev/null
import time; time.sleep(2)

# Download cloudflared if needed
import os.path
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# Start Streamlit in the background
!streamlit run app.py &>/content/st_log.txt &
time.sleep(6)

# Open the tunnel
!nohup ./cloudflared tunnel --url http://localhost:8501 &>/content/cf_log.txt &
time.sleep(8)

# Print the public URL
!grep -o 'https://[-a-z0-9]*\.trycloudflare\.com' /content/cf_log.txt | head -1

^C
^C
https://addressing-las-views-turbo.trycloudflare.com
